# تحويل النموذج لصيغة الموبايل (GGUF) — Convert the model for the phone app

النوتبوك دي بتعمل 3 حاجات ورا بعض:
1. **دمج** الـ adapter مع النموذج الأساسي (زي ما حصل في نوتبوك التشغيل).
2. **تحويل وضغط** النموذج لصيغة GGUF ‏(Q4_K_M — حوالي 950MB) اللي بتشتغل
   على معالج الموبايل جوه التطبيق نفسه، من غير سيرفر ولا رابط.
3. **رفع** الملف النهائي لحسابك على Hugging Face باسم
   `darklink56/qwen-coder-1.5b-gguf` — التطبيق هينزّله من هناك مرة واحدة
   أول تشغيل وبعدها يشتغل أوفلاين على طول.

المتطلبات: توكن Hugging Face بتاعك (نفس توكن التدريب، صلاحية Write) —
هيتطلب في خلية الدخول زي كل مرة. الوقت المتوقع: 15–25 دقيقة.
الـ Runtime على T4 بيخلي الدمج أسرع، بس CPU بيشتغل برضه.


In [ ]:
# 1) Log in to Hugging Face — نفس خلية كل مرة
import os

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

from huggingface_hub import login
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)
    print("✅ Logged in via Colab Secrets (HF_TOKEN)")
else:
    print("ℹ️ هيظهر لك خانة توكن — الصق توكن الـ Write بتاعك:")
    login()
    print("✅ Logged in interactively")


In [ ]:
# 2) Install — نفس إصدارات التدريب المثبتة (من غير bitsandbytes)
!pip install -q "transformers==4.57.6" "peft==0.22.0" "accelerate==1.16.0"
print("✅ Installed")


In [ ]:
# 3) Merge the adapter into the base model and save
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER = "darklink56/qwen-coder-1.5b-lora-out"

on_gpu = torch.cuda.is_available()
dtype = torch.float16 if on_gpu else torch.float32
print("Device:", "GPU" if on_gpu else "CPU", "| dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
tokenizer.pad_token = "<|endoftext|>"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype)
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()
model.save_pretrained("merged-model")
tokenizer.save_pretrained("merged-model")
print("✅ Merged model saved to merged-model/")


In [ ]:
# 4) Get llama.cpp (نفس الإصدار اللي التطبيق مبني بيه: b6500) + build tools
!git clone --depth 1 --branch b6500 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!cmake --version || apt-get -qq install -y cmake
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --target llama-quantize llama-cli -j 4
print("✅ llama.cpp tools built")


In [ ]:
# 5) Convert HF -> GGUF (f16) then quantize to Q4_K_M (حجم الموبايل)
!python3 llama.cpp/convert_hf_to_gguf.py merged-model --outtype f16 --outfile model-f16.gguf
!./llama.cpp/build/bin/llama-quantize model-f16.gguf qwen-coder-1.5b-q4_k_m.gguf Q4_K_M

import os
size_mb = os.path.getsize("qwen-coder-1.5b-q4_k_m.gguf") / 1e6
print(f"✅ GGUF ready: qwen-coder-1.5b-q4_k_m.gguf ({size_mb:.0f} MB)")


In [ ]:
# 6) Smoke test — نتأكد إن الملف سليم وبيولّد كود قبل الرفع
!echo "Write a Python function that adds two numbers." | ./llama.cpp/build/bin/llama-cli -m qwen-coder-1.5b-q4_k_m.gguf -p "<|im_start|>user\nWrite a Python function that adds two numbers.<|im_end|>\n<|im_start|>assistant\n" -n 60 --temp 0 -no-cnv 2>/dev/null


In [ ]:
# 7) Upload the GGUF to your Hugging Face
from huggingface_hub import HfApi

REPO = "darklink56/qwen-coder-1.5b-gguf"
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_file(
    path_or_fileobj="qwen-coder-1.5b-q4_k_m.gguf",
    path_in_repo="qwen-coder-1.5b-q4_k_m.gguf",
    repo_id=REPO,
)
print("🎉 Uploaded!")
print("رابط التنزيل اللي التطبيق هيستخدمه:")
print(f"https://huggingface.co/{REPO}/resolve/main/qwen-coder-1.5b-q4_k_m.gguf")
print("ابعته لـ Pip في الشات عشان يأكد إن كل حاجة تمام.")
